In [1]:
from typing_extensions import TypedDict
from typing import List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model(model="llama3.1:8B", model_provider="ollama")

In [2]:
class State(TypedDict):
    dish: str
    ingredients: list[dict]
    recipe_steps: str
    plating_instructions: str

class Ingredient(BaseModel):
    name: str
    quantity: str
    unit: str

class IngredientsOutput(BaseModel):
    ingredients: List[Ingredient]

In [3]:
def list_ingredients(state: State):
    structured_llm = llm.with_structured_output(IngredientsOutput)
    response = structured_llm.invoke(
        f"List 5-8 ingredients needed to make {state["dish"]}"
    )
    return {"ingredients": response.ingredients}


def create_recipe(state: State):
    response = llm.invoke(
        f"Write a step by step cooking instruction for {state["dish"]}, using these ingredients {state['ingredients']}"
    )
    return {"recipe_steps": response.content}


def describe_plating(state: State):
    response = llm.invoke(
        f"Describe how to beautifully plate this dish {state["dish"]} based on this recipe {state["recipe_steps"]}"
    )
    return {"plating_instructions": response.content}

In [4]:
graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe", create_recipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_edge("list_ingredients", "create_recipe")
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

In [5]:
graph.invoke({ "dish": "hummus" })

{'dish': 'hummus',
 'ingredients': [Ingredient(name='chickpeas', quantity='1 cup, drained and rinsed', unit=''),
  Ingredient(name='tahini', quantity='2 tablespoons', unit=''),
  Ingredient(name='lemon juice', quantity='type of measurement: tablespoon(s) or juice', unit=''),
  Ingredient(name='garlic', quantity='1-2 cloves, minced', unit=''),
  Ingredient(name='olive oil', quantity='1/4 cup', unit='')],
 'recipe_steps': "Here's a step-by-step cooking instruction for hummus using the given ingredients:\n\n**Step 1: Drain and Rinse the Chickpeas**\nDrain the liquid from the chickpea can and rinse the chickpeas with cold water. This helps to remove any excess sodium and impurities.\n\n**Step 2: Peel and Mince the Garlic**\nPeel the garlic cloves and mince them using a garlic press or a microplane grater. You can use either 1 or 2 cloves, depending on your desired level of garlic flavor.\n\n**Step 3: Combine Chickpeas, Tahini, Lemon Juice, and Garlic in a Blender**\nAdd the drained and rin